Importing necessary libraries

In [72]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from urllib.parse import quote_plus
from sqlalchemy import create_engine
from matplotlib import pyplot as plt

Data_Loading

In [73]:
df = pd.read_csv(r"C:\Users\Salman\Data science\Luxury_Housing_analysis\Luxury_Housing_Bangalore.csv")
display(df.head(5))
df.shape

,Property_ID,Micro_Market,Project_Name,Developer_Name,Unit_Size_Sqft,Configuration,Ticket_Price_Cr,Transaction_Type,Buyer_Type,Purchase_Quarter,Connectivity_Score,Amenity_Score,Possession_Status,Sales_Channel,NRI_Buyer,Locality_Infra_Score,Avg_Traffic_Time_Min,Buyer_Comments
0,PROP000001,Sarjapur Road,Project_0,RMZ,4025.0,4bhk,12.750846039118798,Primary,NRI,2025-03-31,7.990091,5.462863,Launch,Broker,yes,9.212491,18,Loved the amenities!
1,PROP000002,Indiranagar,Project_1,Puravankara,5760.0,3Bhk,16.292151871065954,Primary,Other,2024-06-30,4.839024,NaN,Under construction,NRI Desk,no,7.723898,106,NaN
2,PROP000003,Bannerghatta Road,Project_2,Tata Housing,7707.0,4bhk,10.517724412961911,Primary,HNI,2023-12-31,8.131315,8.669227,Ready to move,Direct,yes,6.985493,113,Agent was not responsive.
3,PROP000004,bellary road,Project_3,Embassy,6192.0,3BHK,9.396367494232896,Primary,HNI,2024-03-31,7.501657,5.720246,Ready to move,Online,yes,6.100929,106,Excellent location!
4,PROP000005,Koramangala,Project_4,SNN Raj,7147.0,4Bhk,15.345392444511946,Secondary,HNI,2024-12-31,4.525216,8.609649,Under construction,Broker,no,5.312510,18,Too far from my office.


(101000, 18)

Data cleaning and preprocessing

In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101000 entries, 0 to 100999
Data columns (total 18 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Property_ID           101000 non-null  str    
 1   Micro_Market          101000 non-null  str    
 2   Project_Name          101000 non-null  str    
 3   Developer_Name        101000 non-null  str    
 4   Unit_Size_Sqft        90954 non-null   float64
 5   Configuration         101000 non-null  str    
 6   Ticket_Price_Cr       90981 non-null   str    
 7   Transaction_Type      101000 non-null  str    
 8   Buyer_Type            101000 non-null  str    
 9   Purchase_Quarter      101000 non-null  str    
 10  Connectivity_Score    101000 non-null  float64
 11  Amenity_Score         90910 non-null   float64
 12  Possession_Status     101000 non-null  str    
 13  Sales_Channel         101000 non-null  str    
 14  NRI_Buyer             101000 non-null  str    
 15  Locality_In

In [75]:
for col in df:
    print(col)
    print(df[col].value_counts())

Property_ID
Property_ID
PROP000051    2
PROP000151    2
PROP000157    2
PROP000233    2
PROP000325    2
             ..
PROP099996    1
PROP099997    1
PROP099998    1
PROP099999    1
PROP100000    1
Name: count, Length: 100000, dtype: int64
Micro_Market
Micro_Market
Jayanagar            2176
Jp Nagar             2167
Bannerghatta Road    2164
jp nagar             2163
mg road              2160
domlur               2156
sarjapur road        2152
indiranagar          2149
bannerghatta road    2149
SARJAPUR ROAD        2149
WHITEFIELD           2146
Sarjapur Road        2134
jayanagar            2131
bellary road         2122
JAYANAGAR            2120
yelahanka            2119
MG ROAD              2118
BANNERGHATTA ROAD    2114
Yelahanka            2114
JP NAGAR             2111
Electronic City      2110
Kanakapura Road      2108
kanakapura road      2106
hennur road          2105
RAJAJINAGAR          2105
Whitefield           2104
Hebbal               2102
ELECTRONIC CITY      2094
Indi

In [79]:
df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].str.replace("Cr", "").str.replace("₹", "").str.strip().astype(float)

In [80]:
df['Purchase_Quarter'] = pd.to_datetime(df['Purchase_Quarter'])

In [81]:
df['Micro_Market'].unique()

<StringArray>
[    'Sarjapur Road',       'Indiranagar', 'Bannerghatta Road',
      'bellary road',       'Koramangala',         'YELAHANKA',
        'whitefield',     'sarjapur road',          'JP NAGAR',
       'Rajajinagar',       'koramangala',        'WHITEFIELD',
       'indiranagar',         'jayanagar',        'Whitefield',
            'DOMLUR',         'yelahanka',           'Mg Road',
       'HENNUR ROAD',         'Jayanagar',            'Domlur',
   'Electronic City',            'Hebbal',           'mg road',
          'jp nagar', 'BANNERGHATTA ROAD',           'MG ROAD',
   'KANAKAPURA ROAD',            'HEBBAL',      'BELLARY ROAD',
   'electronic city',   'ELECTRONIC CITY',      'Bellary Road',
       'Hennur Road',       'rajajinagar',   'Kanakapura Road',
       'INDIRANAGAR',       'hennur road',   'kanakapura road',
            'domlur', 'bannerghatta road',       'RAJAJINAGAR',
            'hebbal',         'Yelahanka',     'SARJAPUR ROAD',
       'KORAMANGALA',     

In [82]:
df['Micro_Market'] = df['Micro_Market'].str.strip().str.title()

In [83]:
df['Configuration'] = df['Configuration'].replace({
    '3Bhk' : '3BHK',
    '3bhk' : '3BHK',
    '4bhk' : '4BHK',
    '4Bhk' : '4BHK',
    '5bhk+' : '5BHK+',
    '5Bhk+' : '5BHK+'
})

In [84]:
df['Property_ID'].duplicated().sum()

np.int64(1000)

In [85]:
df = df.drop_duplicates(subset='Property_ID').reset_index(drop=True)

Function to find outliers

In [53]:
def outlier(df):
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1
    upper_limit = Q3 + IQR
    lower_limit = Q1 - IQR
    return upper_limit, lower_limit
outlier(df['Unit_Size_Sqft'])

(np.float64(10516.0), np.float64(1459.0))

In [86]:
outlier(df['Ticket_Price_Cr'])

(np.float64(18.183049122778808), np.float64(5.92492948966375))

In [97]:
len(df[(df['Ticket_Price_Cr']<5.29) | (df['Ticket_Price_Cr']>18.18)])

3784

In [96]:
len(df[(df['Unit_Size_Sqft']<1459) | (df['Unit_Size_Sqft']>10516)])

500

In [100]:
df[df['Unit_Size_Sqft']==-1]

,Property_ID,Micro_Market,Project_Name,Developer_Name,Unit_Size_Sqft,Configuration,Ticket_Price_Cr,Transaction_Type,Buyer_Type,Purchase_Quarter,Connectivity_Score,Amenity_Score,Possession_Status,Sales_Channel,NRI_Buyer,Locality_Infra_Score,Avg_Traffic_Time_Min,Buyer_Comments
145,PROP000146,Bellary Road,Project_145,Puravankara,-1.0,3BHK,9.094891,Secondary,Startup Founder,2024-03-31,9.952351,7.824742,Ready to move,Direct,yes,6.690428,78,Loved the amenities!
238,PROP000239,Koramangala,Project_238,Sobha,-1.0,5BHK+,11.065758,Secondary,Other,2024-06-30,8.900902,NaN,Ready to move,Broker,yes,5.698885,43,Too far from my office.
244,PROP000245,Sarjapur Road,Project_244,L&T Realty,-1.0,5BHK+,18.426594,Secondary,Startup Founder,2023-09-30,5.045976,7.885277,Under construction,Online,yes,5.766082,102,Connectivity is poor.
289,PROP000290,Yelahanka,Project_289,Sobha,-1.0,4BHK,12.199936,Secondary,CXO,2024-06-30,7.194905,6.825103,Ready to move,NRI Desk,no,9.728299,21,Loved the amenities!
747,PROP000748,Sarjapur Road,Project_247,Embassy,-1.0,3BHK,9.448331,Primary,Other,2023-12-31,8.701167,5.982552,Ready to move,Online,yes,8.384907,57,Loved the amenities!
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99443,PROP099444,Kanakapura Road,Project_443,Godrej,-1.0,4BHK,10.099733,Secondary,CXO,2024-06-30,4.681994,9.818244,Ready to move,Direct,no,6.934660,105,Agent was not responsive.
99483,PROP099484,Electronic City,Project_483,Total Environment,-1.0,4BHK,10.524345,Secondary,Other,2024-12-31,8.882498,5.819123,Launch,Direct,yes,7.838773,46,Great value for money.
99494,PROP099495,Sarjapur Road,Project_494,L&T Realty,-1.0,3BHK,12.019192,Secondary,NRI,2024-09-30,5.235146,5.278620,Ready to move,Online,yes,9.560632,85,Great view from 15th floor.
99754,PROP099755,Electronic City,Project_254,Tata Housing,-1.0,3BHK,13.815716,Secondary,Startup Founder,2023-12-31,5.893515,7.898360,Launch,Broker,yes,9.109356,93,Great value for money.


In [103]:
df[(df['Ticket_Price_Cr']<1)]

,Property_ID,Micro_Market,Project_Name,Developer_Name,Unit_Size_Sqft,Configuration,Ticket_Price_Cr,Transaction_Type,Buyer_Type,Purchase_Quarter,Connectivity_Score,Amenity_Score,Possession_Status,Sales_Channel,NRI_Buyer,Locality_Infra_Score,Avg_Traffic_Time_Min,Buyer_Comments
3645,PROP003646,Indiranagar,Project_145,Embassy,5092.0,4BHK,-0.391623,Primary,Startup Founder,2024-06-30,7.149634,6.123946,Under construction,Direct,no,6.967136,105,Too far from my office.
5261,PROP005262,Kanakapura Road,Project_261,Sobha,7533.0,5BHK+,0.750000,Secondary,Startup Founder,2024-03-31,5.736754,8.920321,Launch,NRI Desk,no,5.310623,65,Great value for money.
6130,PROP006131,Hennur Road,Project_130,RMZ,4019.0,5BHK+,-1.420000,Primary,Startup Founder,2023-06-30,6.344294,6.345932,Launch,Direct,no,5.137908,40,Underpriced for location.
7661,PROP007662,Yelahanka,Project_161,Sobha,6573.0,3BHK,0.776524,Secondary,Startup Founder,2024-06-30,5.168312,9.513074,Under construction,Online,no,8.383833,79,NaN
16854,PROP016855,Whitefield,Project_354,L&T Realty,3816.0,4BHK,0.784637,Primary,CXO,2024-06-30,5.481577,6.053821,Ready to move,Online,yes,9.304972,115,NaN
39358,PROP039359,Hebbal,Project_358,Tata Housing,4924.0,5BHK+,-0.287197,Primary,NRI,2023-09-30,4.167448,NaN,Ready to move,NRI Desk,yes,8.249373,115,NaN
50185,PROP050186,Jp Nagar,Project_185,Godrej,5807.0,5BHK+,-0.193873,Primary,CXO,2024-12-31,5.707500,9.986799,Launch,Broker,no,7.867063,73,Excellent location!
74563,PROP074564,Yelahanka,Project_63,Brigade,6162.0,3BHK,0.660000,Secondary,HNI,2023-06-30,8.192522,5.203869,Ready to move,Broker,yes,5.831104,90,Underpriced for location.
78312,PROP078313,Domlur,Project_312,Sobha,4721.0,4BHK,0.955218,Secondary,Startup Founder,2024-12-31,6.243554,9.392875,Under construction,Broker,no,9.139584,24,NaN
83801,PROP083802,Hebbal,Project_301,Tata Housing,6548.0,3BHK,-0.519437,Primary,HNI,2025-03-31,9.180127,7.053654,Under construction,Broker,no,8.870240,87,Great view from 15th floor.


Filling null values and outliers

In [57]:
df['Unit_Size_Sqft'] = df['Unit_Size_Sqft'].replace(-1.0, np.nan)
df['Unit_Size_Sqft'] = df['Unit_Size_Sqft'].fillna(df.groupby(['Developer_Name','Configuration'])['Unit_Size_Sqft'].transform('mean'))

In [58]:
df.loc[df['Ticket_Price_Cr'] < 0, 'Ticket_Price_Cr'] = np.nan

In [59]:
df.loc[df['Ticket_Price_Cr'] > 20, 'Ticket_Price_Cr'] = np.nan

In [60]:
df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].fillna(df.groupby(['Unit_Size_Sqft','Configuration','Transaction_Type'])['Ticket_Price_Cr'].transform('mean'))
df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].fillna(
    df.groupby(['Configuration','Transaction_Type'])['Ticket_Price_Cr'].transform('mean')
)

In [61]:
df['Amenity_Score'] = df['Amenity_Score'].fillna(
    df.groupby(['Developer_Name','Configuration'])['Amenity_Score'].transform('mean')
)

In [62]:
df['Buyer_Comments'] = df['Buyer_Comments'].fillna(df['Buyer_Comments'].mode()[0])

Creating new columns (Price_per_Sqft, Quarter_Number, Booking_Flag)

In [63]:
df['Price_per_Sqft'] = df['Ticket_Price_Cr'] * 10000000 / df['Unit_Size_Sqft']

In [64]:
df['Quarter_Number'] = df['Purchase_Quarter'].dt.quarter

In [65]:
df['Booking_Flag'] = 1

In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Property_ID           100000 non-null  str           
 1   Micro_Market          100000 non-null  str           
 2   Project_Name          100000 non-null  str           
 3   Developer_Name        100000 non-null  str           
 4   Unit_Size_Sqft        100000 non-null  float64       
 5   Configuration         100000 non-null  str           
 6   Ticket_Price_Cr       100000 non-null  float64       
 7   Transaction_Type      100000 non-null  str           
 8   Buyer_Type            100000 non-null  str           
 9   Purchase_Quarter      100000 non-null  datetime64[us]
 10  Connectivity_Score    100000 non-null  float64       
 11  Amenity_Score         100000 non-null  float64       
 12  Possession_Status     100000 non-null  str           
 13  Sales_Chann

In [67]:
df.describe()

,Unit_Size_Sqft,Ticket_Price_Cr,Purchase_Quarter,Connectivity_Score,Amenity_Score,Locality_Infra_Score,Avg_Traffic_Time_Min,Price_per_Sqft,Quarter_Number,Booking_Flag
count,100000.000000,100000.000000,100000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.0
mean,6005.402569,11.962868,2024-05-15 06:37:03.072000,6.993001,7.504224,7.499378,67.188020,21674.736670,2.497820,1.0
min,3000.000000,0.660000,2023-06-30 00:00:00,4.000031,5.000224,5.000013,15.000000,995.619275,1.000000,1.0
25%,4683.000000,10.098588,2023-09-30 00:00:00,5.495535,6.395876,6.249147,41.000000,15535.997476,1.000000,1.0
50%,6009.260966,11.960000,2024-03-31 00:00:00,6.986316,7.499913,7.497347,67.000000,19887.225675,2.000000,1.0
75%,7332.000000,13.855399,2024-09-30 00:00:00,8.490617,8.615671,8.751793,93.000000,26158.352117,3.000000,1.0
max,8999.000000,19.995758,2025-03-31 00:00:00,9.999970,9.999865,9.999956,119.000000,64575.508663,4.000000,1.0
std,1638.300014,2.850045,NaN,1.731699,1.366666,1.443286,30.267763,8634.266681,1.117286,0.0


Establishing database connection and transferring cleaned dataframe to SQL as Table

In [68]:
user = "shaik"
password = "Password@000"
host = "localhost"
port = 3306
db = "luxury_housing_price_analysis"
autocommit = True
encoded_password = quote_plus(password)

engine = create_engine(
    f"mysql+pymysql://{user}:{encoded_password}@{host}:{port}/{db}"
)

def to_sql(dataframe, table_name):
    try:
        dataframe.to_sql(
            name=table_name,
            con=engine,
            index=False,
            if_exists='fail'
        )
        print("First run: Table created and data uploaded successfully!")

    except ValueError:
        print("Table already exists. Skipping upload to keep original data.")

In [69]:
to_sql(df , 'luxury_housing_data')

Table already exists. Skipping upload to keep original data.
